# Universidade Federal de São João del-Rei

# Machine Learning Aplicado à Engenharia

# Trabalho 2 (T2) — Modelagem, Validação e Comparação de Modelos de Aprendizado de Máquina

**Tema:** Análise Exploratória de Sinais EEG para Monitoramento e Classificação de Estados de Sono em Janelas de 30 Segundos

**Base utilizada:** _Sleep-EDF Database Expanded_

**Fonte:** https://www.physionet.org/content/sleep-edfx/1.0.0/

# Referência da Base

- B Kemp, AH Zwinderman, B Tuk, HAC Kamphuisen, JJL Oberyé. _Analysis of a sleep-dependent neuronal feedback loop: the slow-wave microcontinuity of the EEG_. IEEE-BME 47(9):1185-1194 (2000).

# Citação Padrão para PhysioNet

- Goldberger, A., Amaral, L., Glass, L., Hausdorff, J., Ivanov, PC, Mark, R., ... & Stanley, HE (2000). PhysioBank, PhysioToolkit, and PhysioNet: Components of a new research resource for complex physiologic signals. Circulation [Online]. 101 (23), pp. e215–e220. RRID:SCR_007345.

**Integrantes do grupo:**
- Ícaro Ferreira dos Santos
- Wendell Christian Calixto Pinto
- Gabriel Augusto Silva Batista

**Repositório do grupo:**  
https://github.com/gabrielbtt/analise-estados-sono-eeg

# Convenção de Cores para Identificar Contribuições
- <font color="blue">Azul:</font> Ícaro Ferreira dos Santos
- <font color="green">Verde:</font> Wendell Christian Calixto Pinto
- <font color="red">Vermelho:</font> Gabriel Augusto Silva Batista

# Descrição da Base de Dados

A base utilizada é a Sleep-EDF Database Expanded, obtida via PhysioNet. Trata-se de um conjunto de dados reais de polissonografia voltado para o estudo da arquitetura do sono.

Os dados dos sinais polissonográficos (arquivos com extensão .PSG.edf) estão no formato EDF (European Data Format*), contendo EEG (eletrodos Fpz-Cz e Pz-Oz), EOG (horizontal), EMG submentoniano, um marcador de evento e, geralmente, também sinais de respiração oronasal e temperatura corporal retal. Os hipnogramas, anotações dos estágios do sono, armazenados em arquivos com extensão *.Hypnogram.edf, estão no formato EDF+. Os arquivos são nomeados no padrão SC4 ssN EO-PSG.edf, em que "ss" corresponde ao número do sujeito e "N" à noite de coleta.

**Sinais Utilizados:** Foram extraídas características dos canais de EEG Fpz-Cz e Pz-Oz, com frequência de amostragem de 100 Hz.

**Anotações:** Os hipnogramas foram anotados manualmente por técnicos treinados seguindo o padrão Rechtschaffen & Kales, dividindo o registro em épocas de 30 segundos.

**Classes de Sono:** O problema consiste em uma classificação multiclasse com 5 categorias discretas:

- W (Vigília): momento em que a pessoa está acordada ou quase acordando;
- N1: sono leve de transição (fácil de acordar);
- N2: sono mais estável, que ocupa a maior parte da noite;
- N3: sono profundo, importante para recuperação física do corpo;
- R (REM): sono dos sonhos, essencial para a memória e o equilíbrio emocional.

**Escopo do Trabalho:** Para este trabalho, foram utilizados os arquivos no intervalo entre `SC4001E0-PSG.edf`até `SC4012EC-Hypnogram.edf`. Essa expansão permite que o modelo seja treinado e testado em um volume maior de épocas e pacientes em comparação ao estudo exploratório inicial, que foi utilizada apenas uma noite de um paciente.

# Instruções para Obtenção dos Dados

Devido ao grande volume de dados da base **Sleep-EDF Database Expanded** e às restrições de tamanho de arquivo do GitHub, os arquivos brutos não foram incluídos neste repositório. Para que este notebook seja executado corretamente e os resultados sejam reproduzidos, siga as instruções abaixo.

### 1. Clonar o repositório
No próprio notebook, execute o seguinte comando para clonar o repositório do projeto:

> **Nota:** Após a execução do comando abaixo, o ambiente do Notebook exibirá na aba Files, o diretório `analise-estados-sono-eeg` contendo toda a estrutura do repositório clonado.

In [1]:
# Clonar o repositório
!git clone https://github.com/gabrielbtt/analise-estados-sono-eeg.git

Cloning into 'analise-estados-sono-eeg'...
remote: Enumerating objects: 409, done.
remote: Counting objects: 100% (226/226), done.
remote: Compressing objects: 100% (206/206), done.
remote: Total 409 (delta 142), reused 18 (delta 18), pack-reused 183 (from 1)
Receiving objects: 100% (409/409), 97.50 MiB | 17.92 MiB/s, done.
Resolving deltas: 100% (199/199), done.


### 2. Acesse a fonte oficial
Acesse o repositório oficial da base de dados no PhysioNet:

- **[PhysioNet - Sleep-EDF Database Expanded (v1.0.0)](https://www.physionet.org/content/sleep-edfx/1.0.0/#files-panel)**

### 3. Localize os arquivos
Role a página até o final do site. No painel de arquivos (*Files*), localize e navegue até a pasta `sleep-cassette`.

### 4. Realize o download
Baixe os **pares de arquivos `.edf`** correspondentes a cada registro:
- Arquivo de sinal PSG (`*-PSG.edf`)
- Arquivo de hipnograma (`*-Hypnogram.edf`)

> **Nota:** Para este trabalho, foram utilizados os arquivos no intervalo de `SC4001E0-PSG.edf` até `SC4012EC-Hypnogram.edf`.

### 5. Organize o diretório
Salve **obrigatoriamente** todos os arquivos baixados no seguinte caminho, para garantir o funcionamento correto do código de carregamento:

- analise-estados-sono-eeg/dados/edfs_originais/


Após essa organização, o notebook poderá ser executado normalmente e os resultados poderão ser reproduzidos.

# Carregamento da Base de Dados

In [ ]:
# Entrar na pasta do projeto
%cd analise-estados-sono-eeg

In [ ]:
# Verificar se os arquivos estão lá
!dir

In [ ]:
%pip install mne

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import mne
from pathlib import Path
from scipy.signal import welch
from scipy.stats import skew, kurtosis
from scipy.integrate import trapezoid
from IPython.display import display

# Carregamento da Base de Dados
# Pasta com os arquivos EDF originais
dados_dir = Path("dados/edfs_originais")
# Pasta onde o CSV final sera salvo
saida_dir = Path("dados/brutos")
dados_dir

# Leitura e organização
# Extrai a chave comum usada para juntar PSG e hypnograma
def peganome(filename: str) -> str:
    base = Path(filename).name.split("-")[0]
    return base[:7]

# Procura os arquivos e monta os pares correspondentes
def encontrapar(data_dir: Path, prefix="SC"):
    arquivospsg = sorted(data_dir.glob(f"{prefix}*-PSG.edf"))
    arquivoshyp = sorted(data_dir.glob(f"{prefix}*-Hypnogram.edf"))
    # dicionarios para buscar os arquivos pela chave do paciente/noite
    dicpsg = {peganome(arquivo.name): arquivo for arquivo in arquivospsg}
    dichyp = {peganome(arquivo.name): arquivo for arquivo in arquivoshyp}
    # mantem apenas as chaves que possuem os dois arquivos
    dicjunto = sorted(set(dicpsg) & set(dichyp))
    return [(k, dicpsg[k], dichyp[k]) for k in dicjunto]

# Gera a lista de pares validos
pares = encontrapar(dados_dir, prefix="SC")
len(pares), pares[:5]

# Escolher quantidade de noites pra analisar
totalpares = 10
# Seleciona só os primeiros pares para teste
selecpares = pares[:totalpares]
selecpares

# Conferir se par foi carregado corretamente
chavepar, caminhopsg, caminhohyp = selecpares[0]

# le o sinal bruto e as anotacoes do hypnograma
bruto = mne.io.read_raw_edf(caminhopsg, preload=False, infer_types=True, verbose="ERROR")
anot = mne.read_annotations(caminhohyp)

print(f"Chave do par: {chavepar}")
print(f"PSG: {caminhopsg}")
print(f"Hypnograma: {caminhohyp}")
print(f"Canais: {bruto.ch_names}")
print(f"Frequencia de amostragem: {bruto.info['sfreq']} Hz")
print("\nPrimeiras anotacoes:")
# Mostra um resumo inicial para validar as classes e os tempos
print(pd.DataFrame({
    "Inicio(s)": anot.onset[:10],
    "Duracao(s)": anot.duration[:10],
    "Classificacao": anot.description[:10]
}))

# Juntar em blocos de 30s, juntar estágio 3 e 4, selecionar canal principal, juntar com anotações, remover longas partes do acordado, extrair dados
# Tamanho padrão de cada época em segundos
SEGUNDOS_EPOCA = 30.0
# Faixa do filtro passa-banda aplicada ao EEG
FREQ_CORTE_BAIXA = 0.5
FREQ_CORTE_ALTA = 30.0
# Canal principal usado na análise
CANAL_EEG = "Fpz-Cz"
# Pares e pasta final reaproveitados nas próximas células
pares_sel = selecpares
PASTA_SAIDA = saida_dir
PASTA_SAIDA.mkdir(parents=True, exist_ok=True)

# Mapeia os rótulos do hypnograma para ids numéricos
AnotacaoParaId = {
    "Sleep stage W": 1,
    "Sleep stage 1": 2,
    "Sleep stage 2": 3,
    "Sleep stage 3": 4, # juntar estagio 3 e 4
    "Sleep stage 4": 4, # juntar estagio 3 e 4
    "Sleep stage R": 5,
}

# Traduz os ids para os nomes das classes finais
IdPraClassificacao = {
    1: "Acordado",
    2: "N1",
    3: "N2",
    4: "N3",
    5: "REM"
}

# Valida se o canal desejado existe no arquivo lido
def escolher_canal(bruto, escolha="Fpz-Cz"):
    if escolha in bruto.ch_names:
        return escolha
    else:
        raise ValueError(f"Canal {escolha} nao encontrado. Canais disponiveis: {bruto.ch_names}")

# Calcula a potência de uma faixa de frequência usando Welch
def calcular_potencia_banda(x, fs, fmin, fmax):
    amostras_janela = min(len(x), int(4 * fs))
    frequencias, densidade_potencia = welch(x, fs=fs, nperseg=amostras_janela)
    mascara_banda = (frequencias >= fmin) & (frequencias <= fmax)
    if not np.any(mascara_banda):
        return 0.0
    return trapezoid(densidade_potencia[mascara_banda], frequencias[mascara_banda])

# Extrai estatísticas simples e potências por banda de uma época
def extrair_dados_epocas(x, fs):
    delta = calcular_potencia_banda(x, fs=fs, fmin=0.5, fmax=4)
    theta = calcular_potencia_banda(x, fs=fs, fmin=4, fmax=8)
    alpha = calcular_potencia_banda(x, fs=fs, fmin=8, fmax=13)
    beta = calcular_potencia_banda(x, fs=fs, fmin=13, fmax=30)
    total = calcular_potencia_banda(x, fs=fs, fmin=0.5, fmax=30)
    eps = 1e-12
    return {
        "media": np.mean(x),
        "desvio_padrao": np.std(x, ddof=1),
        "variancia": np.var(x, ddof=1),
        "minimo": np.min(x),
        "maximo": np.max(x),
        "pico_a_pico": np.ptp(x),
        "valor_rms": np.sqrt(np.mean(x**2)),
        "assimetria": skew(x, bias=False),
        "curtose_excesso": kurtosis(x, fisher=True, bias=False),
        "potencia_delta": delta,
        "potencia_theta": theta,
        "potencia_alpha": alpha,
        "potencia_beta": beta,
        "potencia_total": total,
        "relativo_delta": delta / (total + eps),
        "relativo_theta": theta / (total + eps),
        "relativo_alpha": alpha / (total + eps),
        "relativo_beta": beta / (total + eps),
        "razao_delta_theta": delta / (theta + eps),
        "razao_delta_alpha": delta / (alpha + eps),
    }

Chave do par: SC4001E
PSG: dados\edfs_originais\SC4001E0-PSG.edf
Hypnograma: dados\edfs_originais\SC4001EC-Hypnogram.edf
Canais: ['Fpz-Cz', 'Pz-Oz', 'horizontal', 'oro-nasal', 'submental', 'rectal', 'Event marker']
Frequencia de amostragem: 100.0 Hz

Primeiras anotacoes:
   Inicio(s)  Duracao(s)  Classificacao
0        0.0     30630.0  Sleep stage W
1    30630.0       120.0  Sleep stage 1
2    30750.0       390.0  Sleep stage 2
3    31140.0        30.0  Sleep stage 3
4    31170.0        30.0  Sleep stage 2
5    31200.0       150.0  Sleep stage 3
6    31350.0        30.0  Sleep stage 4
7    31380.0        60.0  Sleep stage 3
8    31440.0        60.0  Sleep stage 4
9    31500.0        30.0  Sleep stage 3
